> **Notebook-first lesson.** Run cells in order. The final activity is designed to be changed and rerun.

# Lesson 29: Validation, checkpoints and early stopping

A serious training run needs more than a loop that prints loss.

## Validation pass


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += len(yb)

val_acc = correct / total



## Checkpoint


In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": epoch,
    "val_loss": val_loss,
}, "checkpoint.pt")



## Restore


In [ ]:
ckpt = torch.load("checkpoint.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
optimizer.load_state_dict(ckpt["optimizer_state"])



## Track
At minimum record:
- train loss
- validation loss
- chosen metric
- learning rate
- epoch
- seed
- data split
- model configuration

## Exercise
Add early stopping and save only the best validation checkpoint. Then intentionally overfit a tiny dataset and verify your validation curve catches it.


## Runnable activity
Run this experiment. Then change one architectural, data, or optimization choice and compare.

In [ ]:
import torch, tempfile, os
from torch import nn
torch.manual_seed(1)
X=torch.randn(500,2); y=(X[:,0]+.5*X[:,1]>0).long()
Xtr,Xv=X[:350],X[350:]; ytr,yv=y[:350],y[350:]
m=nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,2))
opt=torch.optim.Adam(m.parameters(),lr=.03); loss_fn=nn.CrossEntropyLoss()
best=-1; best_state=None
for epoch in range(100):
    m.train(); opt.zero_grad(); loss=loss_fn(m(Xtr),ytr); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad(): acc=float((m(Xv).argmax(1)==yv).float().mean())
    if acc>best: best=acc; best_state={k:v.detach().clone() for k,v in m.state_dict().items()}
m.load_state_dict(best_state)
print("best validation accuracy",best)

## Explanation checkpoint
Add a Markdown cell that explains the tensor shapes, the mechanism being tested, and what changed when you modified the experiment.